# Baseline Top-K Image Retrieval — Colab Bridge

This notebook is the **single entry point** for running Part 1 on Google Colab.
All model logic lives in the cloned repository; this file only handles
environment setup, configuration, and orchestration.

**Output** → `outputs/<encoder>/` on Colab, then synced to Google Drive  
**Consumed by** → Part 2 CNN Reranking (loads embeddings + Top-20 indices)

```
query_paths[i]  +  gallery_paths[topk_indices[i, j]]  →  (query, candidate) pair
label = int(query_labels[i] == gallery_labels[topk_indices[i, j]])
```

---
**Runtime**: T4 GPU recommended  
**Estimated time**: ~5 min for pretrained CLIP + ResNet; ~20 min if fine-tuning

In [1]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q kaggle
!pip install -q timm
!pip install -q git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done


In [2]:
# ── 2. Mount Google Drive (outputs will be copied here at the end) ────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ── 3. Clone project repository and add to Python path ────────────────────────
REPO_URL  = 'https://github.com/Syndr0/DL_Project.git'
REPO_DIR  = '/content/dl_project'

!git clone {REPO_URL} {REPO_DIR} -q

import sys
sys.path.insert(0, REPO_DIR)
print('Repository cloned and added to sys.path.')

fatal: destination path '/content/dl_project' already exists and is not an empty directory.
Repository cloned and added to sys.path.


In [4]:
# ── 4. Download Kaggle dataset ────────────────────────────────────────────────
# Upload kaggle.json: Kaggle → Settings → API → Create New Token
from google.colab import files
files.upload()   # select kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d iamsouravbanerjee/animal-image-dataset-90-different-animals \
       -p /content/data --unzip -q
print('Dataset ready.')

Saving kaggle.json to kaggle (3).json
Dataset URL: https://www.kaggle.com/datasets/iamsouravbanerjee/animal-image-dataset-90-different-animals
License(s): other
Dataset ready.


In [5]:
# ── 5. Imports ────────────────────────────────────────────────────────────────
import os, random, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from retrieval_engine import (
    build_encoder, fine_tune, extract_all, save_outputs,
    save_metrics_json, submit, build_submission,
)
from baseline.utils.dataset import build_dataset_split
from baseline.utils.metrics import evaluate, precision_at_k

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

ModuleNotFoundError: No module named 'retrieval_engine'

In [ ]:
# ── 6. Build dataset split (70 % gallery / 30 % query, per class) ─────────────
DATA_DIR = '/content/data/animals/animals'

train_paths, train_labels, query_paths, query_labels, classes = \
    build_dataset_split(DATA_DIR, train_ratio=0.7)

print(f'Gallery : {len(train_paths)} images')
print(f'Query   : {len(query_paths)} images')
print(f'Classes : {len(classes)}')

## Primary Encoder 1 — CLIP ViT-B/32
Strong zero-shot baseline. 512-d embeddings.

In [ ]:
# ── 7a. Config ────────────────────────────────────────────────────────────────
BACKBONE1  = 'clip'
VARIANT1   = 'ViT-B/32'
POOL1      = 'gap'    # ignored for CLIP
FINE_TUNE1 = False    # set True to run proxy-classification fine-tuning
FT_EPOCHS  = 10

# ── 7b. Build encoder ─────────────────────────────────────────────────────────
model1, encode1, feat_dim1, tfm1, fwd1 = build_encoder(BACKBONE1, VARIANT1, POOL1)
print(f'feat_dim = {feat_dim1}')

# ── 7c. Optional fine-tuning ──────────────────────────────────────────────────
if FINE_TUNE1:
    print('Fine-tuning CLIP ...')
    t0 = time.time()
    fine_tune(model1, fwd1, feat_dim1, len(classes),
              train_paths, train_labels, tfm1,
              epochs=FT_EPOCHS, backbone=BACKBONE1)
    print(f'Fine-tuning done in {time.time()-t0:.1f}s')

# ── 7d. Extract embeddings ────────────────────────────────────────────────────
t0 = time.time()
gallery_embs1 = extract_all(train_paths, encode1)
query_embs1   = extract_all(query_paths,  encode1)
print(f'Extraction done in {time.time()-t0:.1f}s')

# ── 7e. Evaluate ──────────────────────────────────────────────────────────────
scores1     = evaluate(query_embs1, gallery_embs1, query_labels, train_labels)
precision1  = precision_at_k(query_embs1, gallery_embs1, query_labels, train_labels)
tag1 = f'CLIP ViT-B/32{" [FT]" if FINE_TUNE1 else ""}'
print(f'\n{tag1}')
for k, v in scores1.items():
    print(f'  Recall@{k:2d}   : {v:.4f}    Precision@{k}: {precision1[k]:.4f}')

save_metrics_json(tag1, scores1, precision1, pooling_type=POOL1)

# ── 7f. Save outputs for Part 2 ───────────────────────────────────────────────
out_tag1 = f'{REPO_DIR}/outputs/clip_ViT-B-32_gap{"_ft" if FINE_TUNE1 else ""}'
save_outputs(
    out_tag1,
    gallery_embs1, query_embs1,
    train_paths, query_paths,
    train_labels, query_labels,
    classes,
    topk_k=20,
    eval_scores=scores1,
    precision_scores=precision1,
    metadata={'backbone': BACKBONE1, 'variant': VARIANT1,
              'pool': POOL1, 'fine_tuned': FINE_TUNE1},
)

## Primary Encoder 2 — ResNet-50 GAP
Weaker CNN baseline. Gives Part 2 more room to demonstrate improvement.

In [ ]:
# ── 8a. Config ────────────────────────────────────────────────────────────────
BACKBONE2  = 'resnet'
VARIANT2   = '50'
POOL2      = 'gap'
FINE_TUNE2 = False

# ── 8b. Build + extract + evaluate + save ─────────────────────────────────────
model2, encode2, feat_dim2, tfm2, fwd2 = build_encoder(BACKBONE2, VARIANT2, POOL2)
print(f'feat_dim = {feat_dim2}')

if FINE_TUNE2:
    print('Fine-tuning ResNet-50 (layer4 + MLP head) ...')
    t0 = time.time()
    fine_tune(model2, fwd2, feat_dim2, len(classes),
              train_paths, train_labels, tfm2,
              epochs=FT_EPOCHS, backbone=BACKBONE2)
    print(f'Fine-tuning done in {time.time()-t0:.1f}s')

t0 = time.time()
gallery_embs2 = extract_all(train_paths, encode2)
query_embs2   = extract_all(query_paths,  encode2)
print(f'Extraction done in {time.time()-t0:.1f}s')

scores2    = evaluate(query_embs2, gallery_embs2, query_labels, train_labels)
precision2 = precision_at_k(query_embs2, gallery_embs2, query_labels, train_labels)
tag2 = f'ResNet-50 GAP{" [FT]" if FINE_TUNE2 else ""}'
print(f'\n{tag2}')
for k, v in scores2.items():
    print(f'  Recall@{k:2d}   : {v:.4f}    Precision@{k}: {precision2[k]:.4f}')

save_metrics_json(tag2, scores2, precision2, pooling_type=POOL2)

out_tag2 = f'{REPO_DIR}/outputs/resnet_50_gap{"_ft" if FINE_TUNE2 else ""}'
save_outputs(
    out_tag2,
    gallery_embs2, query_embs2,
    train_paths, query_paths,
    train_labels, query_labels,
    classes,
    topk_k=20,
    eval_scores=scores2,
    precision_scores=precision2,
    metadata={'backbone': BACKBONE2, 'variant': VARIANT2,
              'pool': POOL2, 'fine_tuned': FINE_TUNE2},
)

In [ ]:
# ── 9. Sync outputs to Google Drive ──────────────────────────────────────────
DRIVE_OUT = '/content/drive/MyDrive/dl_project_outputs'
!mkdir -p {DRIVE_OUT}
!cp -r {REPO_DIR}/outputs/. {DRIVE_OUT}/
print(f'Outputs synced to Drive: {DRIVE_OUT}')

In [ ]:
# ── 10. Visualisation ─────────────────────────────────────────────────────────
import torch.nn.functional as F

def show_retrieval(q_idx, gallery_embs, query_embs, g_paths, q_paths,
                   g_labels, q_labels, title='', k=5):
    sims = gallery_embs @ query_embs[q_idx]
    topk_vals, topk_idx = torch.topk(sims, k)
    fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 3))
    axes[0].imshow(Image.open(q_paths[q_idx]))
    axes[0].set_title('Query'); axes[0].axis('off')
    for i, (idx, s) in enumerate(zip(topk_idx.numpy(), topk_vals.numpy())):
        ok = g_labels[idx] == q_labels[q_idx]
        axes[i + 1].imshow(Image.open(g_paths[idx]))
        axes[i + 1].set_title(f"{'V' if ok else 'X'} {s:.3f}",
                               color='green' if ok else 'red', fontsize=10)
        axes[i + 1].axis('off')
    plt.suptitle(title, y=1.02)
    plt.tight_layout(); plt.show()

random.seed(0)
sample_qs = random.sample(range(len(query_paths)), 3)

print('=== CLIP ViT-B/32 ===')
for q in sample_qs:
    show_retrieval(q, gallery_embs1, query_embs1,
                   train_paths, query_paths, train_labels, query_labels,
                   title=f'CLIP ViT-B/32 | Query: {classes[query_labels[q]]}')

print('=== ResNet-50 GAP ===')
for q in sample_qs:
    show_retrieval(q, gallery_embs2, query_embs2,
                   train_paths, query_paths, train_labels, query_labels,
                   title=f'ResNet-50 GAP | Query: {classes[query_labels[q]]}')

In [ ]:
# ── 11. (Optional) Full comparison table — all 17 configs ─────────────────────
# Pretrained-only runs: ~5 min total.
# Fine-tuned rows: ~10 min each. Comment them out to save time.

CONFIGS = [
    ('clip',         'ViT-B/32', 'gap', False),
    ('clip',         'ViT-B/16', 'gap', False),
    ('clip',         'ViT-L/14', 'gap', False),
    ('clip',         'ViT-B/32', 'gap', True),   # fine-tuned
    ('efficientnet', 'b0',       'gap', False),
    ('efficientnet', 'b0',       'gem', False),
    ('efficientnet', 'b3',       'gap', False),
    ('efficientnet', 'b3',       'gem', False),
    ('efficientnet', 'b3',       'gem', True),    # fine-tuned
    ('resnet',       '34',       'gap', False),
    ('resnet',       '50',       'gap', False),
    ('resnet',       '101',      'gap', False),
    ('resnet',       '152',      'gap', False),
    ('resnet',       '152',      'gap', True),    # fine-tuned (layer4 + MLP)
    ('googlenet',    'base',     'gap', False),
    ('googlenet',    'base',     'gem', False),
    ('googlenet',    'base',     'gem', True),    # fine-tuned (last layer)
]

print(f'{"Model":<40} {"R@1":>6} {"R@5":>6} {"R@10":>6} {"P@5":>6}')
print('-' * 65)
for backbone, variant, pool_mode, ft in CONFIGS:
    t0 = time.time()
    m, enc, fd, tr, fw = build_encoder(backbone, variant, pool_mode)
    if ft:
        fine_tune(m, fw, fd, len(classes), train_paths, train_labels,
                  tr, epochs=FT_EPOCHS, backbone=backbone)
    g_embs = extract_all(train_paths, enc)
    q_embs = extract_all(query_paths, enc)
    s      = evaluate(q_embs, g_embs, query_labels, train_labels)
    p      = precision_at_k(q_embs, g_embs, query_labels, train_labels)
    tag    = f'{backbone}/{variant}[{pool_mode}]{" FT" if ft else ""}'
    elapsed = time.time() - t0
    print(f'{tag:<40} {s[1]:>6.4f} {s[5]:>6.4f} {s[10]:>6.4f} {p[5]:>6.4f}  ({elapsed:.0f}s)')
    save_metrics_json(tag, s, p, pooling_type=pool_mode)
    del m, g_embs, q_embs
    torch.cuda.empty_cache()